### Initialization

In [0]:
from pyspark.sql.functions import trim, col

### Read from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101_raw")  
df.display() 

In [0]:
df.printSchema()

### Silver transformations

Rename columns with proper column names.

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "CNTRY": "country"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Remove redundant characters in customer_id.

In [0]:
from pyspark.sql.functions import expr

df = df.withColumn("customer_id", expr("right(customer_id, 5)"))

Check distinct country values.

In [0]:
df.select("country").distinct().show(truncate=False)

Replace "M" with "Male" and "F" with "Female".

In [0]:
df = df.replace(
    "DE",
    "Germany",
    subset=["country"]
)

df = df.replace(
    "US",
    "USA",
    subset=["country"]
)

df = df.replace(
    "United States",
    "USA",
    subset=["country"]
)

df = df.replace(
    "United Kingdom",
    "UK",
    subset=["country"]
)

df.groupBy("country").count().show()

### Write into Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_locations")

In [0]:
%sql
SELECT * 
FROM workspace.silver.erp_locations 
LIMIT 10